In [3]:
from datetime import date

TODAY = date.today().strftime("%Y%m%d")
PREFIX = "483"

CONCORDANCE_URL = (
    "https://map.stockholmarchipelagotrail.com/data/geojson/poi-concordance.json"
)

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

In [8]:
import requests
import time

session = requests.Session()

session.headers.update({
    "User-Agent":
        "SAT-Concordance-Validator/1.0 "
        "(https://github.com/salgo60/Stockholm_Archipelago_Trail; "
        "mailto:YOUR_EMAIL)",
    "Accept": "application/json"
})

In [9]:
def get_json(url, params=None, timeout=180):

    for attempt in range(5):

        r = session.get(
            url,
            params=params,
            timeout=timeout
        )

        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "30"))
            print(f"Rate limited. Sleeping {wait}s")
            time.sleep(wait)
            continue

        r.raise_for_status()

        try:
            return r.json()

        except Exception:
            print(r.text[:500])
            raise

    raise RuntimeError("Too many retries")

In [4]:
import requests
import pandas as pd

def load_concordance():

    data = requests.get(CONCORDANCE_URL).json()

    rows = []

    for external_id, sat_id in data["satIdOf"].items():

        parts = external_id.split(":")

        source = parts[0]

        if source == "osm":

            rows.append({
                "source": "osm",
                "osm_type": parts[1],
                "osm_id": int(parts[2]),
                "wikidata": None,
                "external_id": external_id,
                "sat_id": sat_id
            })

        elif source == "wikidata":

            rows.append({
                "source": "wikidata",
                "osm_type": None,
                "osm_id": None,
                "wikidata": parts[1],
                "external_id": external_id,
                "sat_id": sat_id
            })

        else:

            rows.append({
                "source": source,
                "osm_type": None,
                "osm_id": None,
                "wikidata": None,
                "external_id": external_id,
                "sat_id": sat_id
            })

    return pd.DataFrame(rows)


concordance = load_concordance()

display(concordance.head())

,source,osm_type,osm_id,wikidata,external_id,sat_id
0,grillplatser,None,NaN,None,grillplatser:G-0254e7bf-dd61-49dc-81c4-784e9e7...,sat:poi:eb5j5
1,grillplatser,None,NaN,None,grillplatser:G-052fd3c1-9648-4339-b206-074c5d1...,sat:poi:cc37k
2,grillplatser,None,NaN,None,grillplatser:G-09fc2541-3901-4a50-a68c-5fac553...,sat:poi:fqw8k
3,grillplatser,None,NaN,None,grillplatser:G-0a3e8dd4-6364-4fbd-b6ab-1122266...,sat:poi:459zw
4,grillplatser,None,NaN,None,grillplatser:G-0f556cb2-bd14-4049-a5ad-5b95277...,sat:poi:4f4rw


In [5]:
import requests

query = """
[out:json][timeout:120];
(
  nwr["ref:stockholmarchipelagotrail"];
);
out ids tags;
"""

r = requests.get(
    OVERPASS_URL,
    params={"data": query},
    timeout=180
)

elements = r.json()["elements"]

rows = []

for e in elements:

    rows.append({

        "osm_type": e["type"],
        "osm_id": e["id"],
        "sat_id": e["tags"].get(
            "ref:stockholmarchipelagotrail"
        )
    })

osm = pd.DataFrame(rows)

print(len(osm))

display(osm.head())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [6]:
conc_osm = concordance[
    concordance.source == "osm"
][["sat_id","osm_type","osm_id"]]

compare = conc_osm.merge(
    osm,
    on="sat_id",
    how="outer",
    suffixes=("_conc","_osm")
)

compare["status"] = "OK"

compare.loc[
    compare.osm_id_conc.isna(),
    "status"
] = "Missing in concordance"

compare.loc[
    compare.osm_id_osm.isna(),
    "status"
] = "Missing in OSM"

mask = (
    compare.status == "OK"
) & (
    (compare.osm_id_conc != compare.osm_id_osm) |
    (compare.osm_type_conc != compare.osm_type_osm)
)

compare.loc[
    mask,
    "status"
] = "Mismatch"

compare["osm_link"] = compare.apply(
    lambda r:
    f"https://www.openstreetmap.org/{r.osm_type_osm}/{int(r.osm_id_osm)}"
    if pd.notna(r.osm_id_osm)
    else "",
    axis=1
)

compare.to_csv(
    f"{PREFIX}_concordance_vs_osm_{TODAY}.csv",
    index=False
)

compare

NameError: name 'osm' is not defined

In [7]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper(SPARQL_ENDPOINT)

sparql.setQuery("""
SELECT ?item ?sat WHERE {
  ?item wdt:P14545 ?sat .
}
""")

sparql.setReturnFormat(JSON)

results = sparql.query().convert()

rows = []

for r in results["results"]["bindings"]:

    rows.append({

        "qid":
            r["item"]["value"].split("/")[-1],

        "sat_id":
            r["sat"]["value"]
    })

wd = pd.DataFrame(rows)

display(wd.head())

HTTPError: HTTP Error 429: Aggressively rate-limiting to 1 req / min - this rule was created during active wdqs outage (797a132)